# Nested Booking Limits — Interactive Dashboard
Run the cell below in Google Colab to use the dashboard.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

FARES = [('Promo',79),('Saver',120),('Standard',190),('Flex',320)]
CAPACITY = 180
INITIAL_SOLD = [20,20,20,20]
LIMITS = [60,100,140,180]
sold = INITIAL_SOLD.copy()

style = widgets.HTML(value="""<style>
.rm-wrap{font-family:Arial,sans-serif;max-width:1000px;color:#07324a}.rm-title{font-size:26px;font-weight:700;margin:6px 0}.rm-sub{color:#64748b;font-size:13px;margin-bottom:18px}.rm-kpis{display:flex;gap:12px;margin:12px 0}.rm-kpi{background:#f0f4f8;border-radius:10px;padding:15px 20px;flex:1}.rm-kpi small{color:#64748b}.rm-kpi strong{display:block;font-size:27px;margin-top:5px}.rm-track{height:13px;background:#e5e7eb;border-radius:8px;overflow:hidden;margin:14px 0 22px}.rm-fill{height:100%;background:#086b95}.rm-table{width:100%;border-collapse:collapse;font-size:15px}.rm-table th{text-align:left;background:#edf3f8;padding:12px}.rm-table td{padding:12px;border-bottom:1px solid #e5e7eb}.rm-open{color:#087e4e;font-weight:700}.rm-closed{color:#b91c1c;font-weight:700}.rm-note{color:#64748b;font-size:12px;margin-top:12px}</style>""")
out=widgets.Output()
buttons=[]

def availability():
    remaining = CAPACITY-sum(sold)
    return [max(0,min(remaining,LIMITS[i]-sum(sold[:i+1]))) for i in range(4)]

def render():
    a=availability(); total=sum(sold); remaining=CAPACITY-total
    rows=''.join(f'<tr><td><b>{name}</b><br><small>${fare}</small></td><td>{sold[i]}</td><td>{LIMITS[i]}</td><td><b>{a[i]}</b></td><td class="{"rm-open" if a[i]>0 else "rm-closed"}">{"OPEN" if a[i]>0 else "CLOSED"}</td></tr>' for i,(name,fare) in enumerate(FARES))
    html=f"""<div class="rm-wrap"><div class="rm-sub">ILLUSTRATIVE REVENUE MANAGEMENT DASHBOARD</div><div class="rm-title">Flight Inventory Control</div><div class="rm-sub">180-seat flight · Economy cabin · 14 days before departure</div><div class="rm-kpis"><div class="rm-kpi"><small>Seats sold</small><strong>{total}</strong></div><div class="rm-kpi"><small>Remaining</small><strong>{remaining}</strong></div><div class="rm-kpi"><small>Load factor</small><strong>{total/CAPACITY:.1%}</strong></div></div><div class="rm-track"><div class="rm-fill" style="width:{total/CAPACITY*100:.2f}%"></div></div><table class="rm-table"><thead><tr><th>Fare class</th><th>Sold</th><th>Nested limit</th><th>Available</th><th>Status</th></tr></thead><tbody>{rows}</tbody></table><div class="rm-note">Availability values overlap and must not be added together. This is a simplified cumulative nested-limit model, not a live airline system.</div></div>"""
    with out:
        clear_output(wait=True)
        display(HTML(html))
    for i,b in enumerate(buttons):
        b.disabled = a[i]==0

def book(i):
    if availability()[i]>0:
        sold[i]+=1
        render()

for i,(name,fare) in enumerate(FARES):
    b=widgets.Button(description=f'+1 {name} (${fare})',layout=widgets.Layout(width='190px'),button_style='')
    b.on_click(lambda _,i=i:book(i))
    buttons.append(b)
reset=widgets.Button(description='↻ Reset example',button_style='info',layout=widgets.Layout(width='190px'))
def reset_all(_):
    sold[:]=INITIAL_SOLD
    render()
reset.on_click(reset_all)
display(style,out,widgets.HTML('<b>Simulate a booking</b> — accept one passenger in a fare class:'),widgets.HBox(buttons,layout=widgets.Layout(flex_flow='row wrap',gap='8px')),reset)
render()